# 01 — Compare cloud API baselines without hosting either model

This CPU-only lab prepares the frozen BANKING77 private-routing benchmark and evaluates two NVIDIA-hosted endpoints: the 30B/3B-active Nemotron 3.5 Lightning model that we will fine-tune and the much larger 550B/55B-active Nemotron 3 Ultra reference. It includes fair prompt-only competitors so the fine-tuned model is not compared only with a model that was never shown the private taxonomy. It can run on a laptop, a CPU instance, or a Brev notebook without consuming GPU memory.

**Precision boundary:** both hosted endpoints serve NVFP4 variants. Notebook 02 repeats the benchmark with the pinned Lightning BF16 customization checkpoint because Notebook 03 tunes BF16. The cloud scores are useful task references, but only local BF16 before versus tuned BF16-derived after isolates the LoRA effect.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', ROOT)
print('Artifacts:', ARTIFACTS_DIR)

## 1. API-only preflight and pinned benchmark inputs

No CUDA check is performed. Data preparation downloads only the public BANKING77 source files and the pinned model tokenizer; it does not download the 60+ GB model weights.

API configuration defaults to the tracked public endpoint and model IDs. For private testing, copy `config/api.local.toml.example` to the Git-ignored `config/api.local.toml`; it will be selected automatically and its reports will receive a separate profile prefix. Set `API_PROFILE_OVERRIDE` in the next cell to `'public'` or `'local'` for an immediate, widget-free switch, then rerun that cell and the Section 2 authentication cell. Leave it as `None` for automatic selection. `NEMOTRON_API_PROFILE` remains available for startup-level selection. Never put an API key in either config file.

In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'api'], check=True)

from nemotron_ft_lab.api_config import load_nvidia_api_config

API_PROFILE_OVERRIDE = None  # None = auto; change to 'public' or 'local' and rerun this cell
API_CONFIG = load_nvidia_api_config(ROOT, profile=API_PROFILE_OVERRIDE)
NVIDIA_API_BASE_URL = API_CONFIG.base_url
NVIDIA_API_REQUESTS_PER_MINUTE = API_CONFIG.requests_per_minute
print('API profile:', API_CONFIG.profile_name, f'({API_CONFIG.source_label})')
print('Endpoint:', NVIDIA_API_BASE_URL)
print('Lightning:', API_CONFIG.lightning.model_id, '->', API_CONFIG.lightning.served_variant)
print('Ultra:', API_CONFIG.ultra.model_id, '->', API_CONFIG.ultra.served_variant)
print('Rate:', NVIDIA_API_REQUESTS_PER_MINUTE, 'requests/minute')
print('After switching profiles, rerun the authentication cell in Section 2.')

In [ ]:
DATA_DIR = ARTIFACTS_DIR / 'data/banking77'
subprocess.run([
    sys.executable, 'scripts/prepare_banking77.py',
    '--output-dir', str(DATA_DIR),
], check=True)
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
manifest['counts'], manifest['official_split_policy']

In [ ]:
from nemotron_ft_lab.data import (
    LexicalDemonstrationRetriever, balanced_evaluation_subset,
    build_few_shot_messages, build_messages, build_taxonomy_messages, read_jsonl,
)

eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
training_rows = read_jsonl(DATA_DIR / 'training.jsonl')
label_map = json.loads((DATA_DIR / 'label_map.json').read_text())
API_EXAMPLES_PER_LABEL = 1  # 1 = 77 requests/model; 3 = 231 requests/model
API_EXPERIMENT_PROFILE = 'retrieval'
PROMPT_MODES_BY_PROFILE = {
    'retrieval': ('retrieved_few_shot',),       # 154 total requests by default
    'prompt_only': ('taxonomy', 'retrieved_few_shot'),  # 308 total
    'full': ('opaque_zero_shot', 'taxonomy', 'retrieved_few_shot'),  # 462 total
}
if API_EXPERIMENT_PROFILE not in PROMPT_MODES_BY_PROFILE:
    raise ValueError(f'Unknown API_EXPERIMENT_PROFILE: {API_EXPERIMENT_PROFILE}')
API_PROMPT_MODES = PROMPT_MODES_BY_PROFILE[API_EXPERIMENT_PROFILE]
DEMONSTRATIONS_PER_REQUEST = 5
PROMPT_PROTOCOL_VERSION = 1  # increment if prompt construction changes
api_eval_rows = balanced_evaluation_subset(
    eval_rows, examples_per_label=API_EXAMPLES_PER_LABEL,
)
retriever = LexicalDemonstrationRetriever(training_rows)
prompt_builders = {
    'opaque_zero_shot': build_messages,
    'taxonomy': lambda utterance: build_taxonomy_messages(utterance, label_map),
    'retrieved_few_shot': lambda utterance: build_few_shot_messages(
        utterance, retriever.retrieve(utterance, k=DEMONSTRATIONS_PER_REQUEST),
    ),
}
assert set(manifest['train_example_ids']).isdisjoint(manifest['validation_example_ids'])
assert all(item.startswith('test-') for item in manifest['test_example_ids'])
assert len(api_eval_rows) == 77 * API_EXAMPLES_PER_LABEL
print('Prepared evaluation pool:', len(eval_rows))
print('Prompt conditions:', API_PROMPT_MODES)
print('Cloud requests per model:', len(API_PROMPT_MODES) * len(api_eval_rows))
print('Cloud requests across two models:', 2 * len(API_PROMPT_MODES) * len(api_eval_rows))
print('Minimum paced runtime across two models (minutes):', round(
    2 * len(API_PROMPT_MODES) * len(api_eval_rows) / NVIDIA_API_REQUESTS_PER_MINUTE, 1
))
print('Example:', {key: api_eval_rows[0][key] for key in ('utterance', 'expected', 'label_name')})

## 2. Authenticate without storing the API key

Create a prototype key from NVIDIA's API Catalog. If `NVIDIA_API_KEY` is already in the kernel environment, the client connects immediately. Otherwise, this cell uses Jupyter's built-in `input()` channel instead of an `ipywidgets` control: paste the key at the native prompt and press **Enter**. The key is temporarily visible while entering it, then the prompt is cleared; it is never printed or written to an artifact.

In [ ]:
from openai import OpenAI
from IPython.display import clear_output

api_key = os.environ.get('NVIDIA_API_KEY', '').strip()
used_native_prompt = not api_key
if used_native_prompt:
    api_key = input('Paste NVIDIA API key (visible until Enter), then press Enter: ').strip()
    clear_output(wait=False)
if not api_key:
    raise RuntimeError('An NVIDIA API key is required for the hosted baseline.')

client = OpenAI(
    base_url=NVIDIA_API_BASE_URL, api_key=api_key, max_retries=0,
    timeout=API_CONFIG.timeout_seconds,
)
del api_key
print('NVIDIA API client configured; native input has been cleared.' if used_native_prompt
      else 'NVIDIA API client configured from NVIDIA_API_KEY.')

## 3. Inspect the prompt-only competitors

`opaque_zero_shot` hides the mapping, reproducing the original difficult baseline. `taxonomy` supplies the complete code-to-intent map without test examples. `retrieved_few_shot` supplies five lexically similar examples selected only from the frozen training set. The default profile runs retrieval for both models, keeping the normal workload at 154 calls. Select `prompt_only` for both prompt competitors or `full` for all three conditions.

In [ ]:
CLOUD_MODELS = {
    'lightning': {
        'model_id': API_CONFIG.lightning.model_id,
        'served_variant': API_CONFIG.lightning.served_variant,
    },
    'ultra': {
        'model_id': API_CONFIG.ultra.model_id,
        'served_variant': API_CONFIG.ultra.served_variant,
    },
}
preview_query = api_eval_rows[0]['utterance']
preview_demos = retriever.retrieve(preview_query, k=DEMONSTRATIONS_PER_REQUEST)
print('Query:', preview_query)
print('Retrieved training-only demonstrations:')
for row in preview_demos:
    print(f"  {row['utterance']!r} -> {row['route_code']} ({row['label_name']})")
print('Taxonomy entries available to taxonomy mode:', len(label_map))

RUN_ORDINARY_PROBES = False  # True adds one non-benchmark request per model
if RUN_ORDINARY_PROBES:
    for name, spec in CLOUD_MODELS.items():
        probe = client.chat.completions.create(
            model=spec['model_id'],
            messages=[{'role': 'user', 'content': 'Explain in two sentences why a bank transfer can remain pending.'}],
            temperature=0.0, max_tokens=80,
            extra_body={'chat_template_kwargs': {'enable_thinking': False}},
        )
        print(f'\n{name.title()}:\n{probe.choices[0].message.content}')

In [ ]:
from nemotron_ft_lab.evaluation import (
    generate_nvidia_api_predictions, paired_accuracy_comparison,
    save_report, score_predictions,
)

if 'client' not in globals():
    raise RuntimeError('Configure the NVIDIA API client in Section 2 before running evaluations.')

api_reports = {}
api_report_paths = {}
for name, spec in CLOUD_MODELS.items():
    for prompt_mode in API_PROMPT_MODES:
        condition = f'{name}::{prompt_mode}'
        print(f'\nEvaluating {condition}: {spec["model_id"]}')
        prompt_variant = f'{prompt_mode}_v{PROMPT_PROTOCOL_VERSION}'
        if prompt_mode == 'retrieved_few_shot':
            prompt_variant += f'_{DEMONSTRATIONS_PER_REQUEST}d'
        artifact_model_name = f'{API_CONFIG.artifact_prefix}{name}'
        resume_path = ARTIFACTS_DIR / (
            f'evaluation/api_{artifact_model_name}_nvfp4_predictions_{prompt_variant}_'
            f'{API_EXAMPLES_PER_LABEL}_per_label.jsonl'
        )
        condition_rows = [{**row, 'prompt_mode': prompt_mode} for row in api_eval_rows]
        started = time.perf_counter()
        generated_rows = generate_nvidia_api_predictions(
            client, condition_rows, model=spec['model_id'], resume_path=resume_path,
            message_builder=prompt_builders[prompt_mode],
            requests_per_minute=NVIDIA_API_REQUESTS_PER_MINUTE,
        )
        report = score_predictions(generated_rows)
        report.update({
            'wall_time_seconds': time.perf_counter() - started,
            'endpoint': NVIDIA_API_BASE_URL,
            'api_profile': API_CONFIG.profile_name,
            'served_variant': spec['served_variant'],
            'precision': 'NVFP4',
            'prompt_mode': prompt_mode,
            'prompt_protocol_version': PROMPT_PROTOCOL_VERSION,
            'demonstrations_per_request': (
                DEMONSTRATIONS_PER_REQUEST if prompt_mode == 'retrieved_few_shot' else 0
            ),
            'examples_per_label': API_EXAMPLES_PER_LABEL,
            'endpoint_revision': 'not exposed by the endpoint',
        })
        report_path = ARTIFACTS_DIR / (
            f'evaluation/baseline_api_{artifact_model_name}_nvfp4_{prompt_variant}_'
            f'{API_EXAMPLES_PER_LABEL}_per_label.json'
        )
        save_report(
            report_path, report, model=spec['model_id'],
            run_type=(
                f'hosted-api-{API_CONFIG.profile_name}-{name}-nvfp4-'
                f'{prompt_mode}-baseline'
            ),
        )
        api_reports[condition] = report
        api_report_paths[condition] = report_path

summary_keys = ('n', 'accuracy', 'macro_accuracy', 'valid_code_rate', 'wall_time_seconds')
{name: {key: report[key] for key in summary_keys} for name, report in api_reports.items()}

In [ ]:
for prompt_mode in API_PROMPT_MODES:
    lightning = api_reports[f'lightning::{prompt_mode}']
    ultra = api_reports[f'ultra::{prompt_mode}']
    ultra_vs_lightning = paired_accuracy_comparison(lightning, ultra)
    ultra_vs_lightning.update({
        'prompt_mode': prompt_mode,
        'lightning_accuracy': lightning['accuracy'],
        'ultra_accuracy': ultra['accuracy'],
        'interpretation': 'Ultra minus Lightning on identical hosted NVFP4 requests',
    })
    print(f'\nUltra versus Lightning — {prompt_mode}')
    print(json.dumps(ultra_vs_lightning, indent=2))

for condition, report in api_reports.items():
    print(f'\nFirst {condition} errors:')
    errors = [row for row in report['rows'] if not row['correct']]
    for row in errors[:5]:
        print(f"{row['utterance']!r}\n  expected={row['expected']} ({row['label_name']}) generated={row['generated']!r}\n")

## Result contract

This notebook saves model-, prompt-, and sample-profile-specific reports such as `baseline_api_ultra_nvfp4_retrieved_few_shot_v1_5d_1_per_label.json`. Prompt protocol and demonstration count are part of the filename so resumable caches cannot silently cross conditions. Notebook 03 evaluates tuned Lightning on the same IDs and reports whether specialization beats, trails, or is inconclusive against every available prompt-only target. Its primary LoRA claim still comes from Notebook 02's local BF16 before-score on all 231 held-out rows; that prevents a precision or serving-backend difference from being mislabeled as fine-tuning gain.